<a href="https://colab.research.google.com/github/PxGT09/Projeto_IA_RH/blob/main/corpo_do_projeto_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Estrutura do Projeto em Python

import imaplib
import email
import pdfplumber
import pandas as pd
import re

# --- CONFIGURAÇÕES ---
EMAIL_USER = 'seu-email@gmail.com'
EMAIL_PASS = 'sua-senha-de-app'
IMAP_SERVER = 'imap.gmail.com'

# Requisitos da Vaga (Pode ser editado para qualquer nicho)
REQUISITOS_VAGA = {
    "escolaridade": ["ensino fundamental", "ensino medio"],
    "experiencia": True,
    "cursos": ["informatica"]
}

def extrair_texto_pdf(caminho_pdf):
    texto = ""
    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            texto += pagina.extract_text()
    return texto.lower()

def analisar_curriculo(texto):
    """
    Lógica simples de triagem.
    Para algo mais avançado, pode-se usar spaCy ou GPT-4/Gemini API.
    """
    # Extração básica via Regex (Exemplo para Telefone)
    telefone = re.findall(r'\(?\d{2}\)?\s?\d{4,5}-?\d{4}', texto)

    # Verificação de Requisitos
    match_escolaridade = any(exp in texto for exp in REQUISITOS_VAGA["escolaridade"])
    match_cursos = any(curso in texto for curso in REQUISITOS_VAGA["cursos"])

    status = "Aprovado" if (match_escolaridade and match_cursos) else "Reprovado"

    return {
        "Telefone": telefone[0] if telefone else "Não encontrado",
        "Escolaridade_OK": match_escolaridade,
        "Cursos_OK": match_cursos,
        "Status": status
    }

def processar_emails():
    # Conexão com o servidor de e-mail
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    mail.login(EMAIL_USER, EMAIL_PASS)
    mail.select("inbox")

    # Busca e-mails não lidos
    _, mensagens = mail.search(None, '(UNSEEN)')

    lista_candidatos = []

    for num in mensagens[0].split():
        _, data = mail.fetch(num, '(RFC822)')
        raw_email = data[0][1]
        msg = email.message_from_bytes(raw_email)

        for part in msg.walk():
            if part.get_content_maintype() == 'multipart': continue
            if part.get('Content-Disposition') is None: continue

            filename = part.get_filename()
            if filename and filename.endswith('.pdf'):
                # Salva o currículo temporariamente
                with open(filename, 'wb') as f:
                    f.write(part.get_payload(decode=True))

                # Extrai e Analisa
                texto_cv = extrair_texto_pdf(filename)
                analise = analisar_curriculo(texto_cv)
                analise['Nome_Arquivo'] = filename
                lista_candidatos.append(analise)

    # Exporta para Excel para o RH revisar
    if lista_candidatos:
        df = pd.DataFrame(lista_candidatos)
        df.to_excel("triagem_candidatos.xlsx", index=False)
        print("Triagem concluída! Arquivo 'triagem_candidatos.xlsx' gerado.")
    else:
        print("Nenhum currículo novo encontrado.")

# Executar o robô
# processar_emails()

In [ ]:
#2. Código Integrado com IA (Versão Inteligente)

#Nesta versão, em vez de criarmos regras manuais, enviamos o texto do currículo e os requisitos da vaga para a IA, pedindo que ela retorne um formato estruturado (JSON).

import google.generativeai as genai
import json

# --- CONFIGURAÇÃO DA API ---
# Obtenha sua chave em: https://aistudio.google.com/
genai.configure(api_key="SUA_CHAVE_AQUI")
model = genai.GenerativeModel('gemini-1.5-flash')

def analisar_cv_com_ia(texto_curriculo, requisitos_vaga):
    prompt = f"""
    Você é um recrutador especializado em RH.
    Analise o currículo abaixo e verifique se ele atende aos requisitos da vaga.

    REQUISITOS DA VAGA:
    {requisitos_vaga}

    TEXTO DO CURRÍCULO:
    {texto_curriculo}

    Retorne APENAS um objeto JSON com o seguinte formato:
    {{
        "nome_candidato": "string",
        "contato": "string",
        "escolaridade_compativel": "Sim/Não",
        "experiencia_resumo": "breve descrição",
        "score_compatibilidade": 0 a 100,
        "aprovado_triagem": boolean
    }}
    """

    response = model.generate_content(prompt)

    # Limpeza para garantir que pegamos apenas o JSON da resposta
    try:
        dados = response.text.replace('```json', '').replace('```', '').strip()
        return json.loads(dados)
    except:
        return {"erro": "Falha ao processar resposta da IA"}

# --- EXEMPLO DE USO ---
requisitos = "Auxiliar de Serviços Gerais, ensino fundamental completo, experiência na área."
curriculo_extraido = "João Silva, residente em Brasília. Trabalhei 2 anos com manutenção predial e limpeza."

resultado = analisar_cv_com_ia(curriculo_extraido, requisitos)
print(resultado)

In [ ]:
#1. Camada de Segurança (Anonimização Simples)
#Antes de enviar o texto para a API da IA, podemos usar expressões regulares para "mascarar" dados sensíveis. Isso protege a privacidade do candidato.

import re

def anonimizar_dados(texto):
    # Mascara CPFs (000.000.000-00)
    texto = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '[CPF OCULTADO]', texto)
    # Mascara RG ou números de documentos similares
    texto = re.sub(r'\b\d{7,9}\b', '[DOC OCULTADO]', texto)
    return texto

In [ ]:
#2. Integração com Google Sheets
#Para isso, você usará a biblioteca gspread.

#Requisito: Você precisará de um arquivo JSON de "Credenciais de Serviço" do Google Cloud Console.

import gspread
from oauth2client.service_account import ServiceAccountCredentials

def salvar_no_google_sheets(dados_candidato):
    # Configuração de acesso
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
    creds = ServiceAccountCredentials.from_json_keyfile_name('seu-arquivo-credenciais.json', scope)
    client = gspread.authorize(creds)

    # Abre a planilha pelo nome
    planilha = client.open("Triagem de Candidatos RH").sheet1

    # Prepara a linha para inserir
    linha = [
        dados_candidato.get("nome_candidato"),
        dados_candidato.get("contato"),
        dados_candidato.get("escolaridade_ok"),
        dados_candidato.get("score_compatibilidade"),
        "Aprovado" if dados_candidato.get("aprovado_triagem") else "Reprovado"
    ]

    planilha.append_row(linha)
    print("Dados salvos na nuvem com sucesso!")

In [ ]:
#1. Instalação
pip install streamlit

In [ ]:
#2. Código do Dashboard (app.py)
#Este código cria uma interface onde você define a vaga e a IA processa os arquivos:

import streamlit as st
import pandas as pd
from your_logic_file import analisar_cv_com_ia, extrair_texto_pdf # Importa suas funções

st.set_page_config(page_title="IA Recrutadora Profissional", layout="wide")

st.title("🤖 Portal de Triagem Inteligente")
st.markdown("---")

# Barra lateral para configurações
with st.sidebar:
    st.header("Configuração da Vaga")
    cargo = st.selectbox("Nicho da Vaga", ["Serviços Gerais", "Segurança do Trabalho", "Administrativo", "TI"])
    requisitos_input = st.text_area("Descreva os requisitos detalhados:",
                                   "Ex: Ensino médio, experiência com limpeza pesada, residir em Brasília.")

    st.info("A IA usará esses critérios para filtrar os candidatos.")

# Área principal
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("📤 Upload de Currículos")
    arquivos_enviados = st.file_uploader("Arraste os PDFs aqui", type="pdf", accept_multiple_files=True)

with col2:
    st.subheader("📊 Resultados da Análise")
    if st.button("Iniciar Triagem Inteligente"):
        if arquivos_enviados:
            resultados = []
            progresso = st.progress(0)

            for i, arquivo in enumerate(arquivos_enviados):
                # Processamento
                texto = extrair_texto_pdf(arquivo)
                analise = analisar_cv_com_ia(texto, requisitos_input)
                resultados.append(analise)

                # Atualiza barra de progresso
                progresso.progress((i + 1) / len(arquivos_enviados))

            # Exibe a tabela final
            df = pd.DataFrame(resultados)
            st.dataframe(df.style.highlight_max(axis=0, subset=['score_compatibilidade'], color='#90EE90'))

            # Botão para baixar relatório
            st.download_button("Baixar Planilha de Resultados", df.to_csv(), "triagem.csv", "text/csv")
        else:
            st.warning("Por favor, envie ao menos um currículo.")

In [ ]:
#1. Modelagem do Banco (SQL)
#Vamos criar duas tabelas principais: uma para as Vagas e outra para os Candidatos.

import sqlite3

def criar_banco():
    conn = sqlite3.connect('rh_inteligente.db')
    cursor = conn.cursor()

    # Tabela de Vagas
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS vagas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            requisitos TEXT,
            data_criacao DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    # Tabela de Candidatos (com FK para Vaga)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS candidatos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            vaga_id INTEGER,
            nome TEXT,
            contato TEXT,
            escolaridade TEXT,
            score INTEGER,
            resumo_ia TEXT,
            status TEXT,
            FOREIGN KEY (vaga_id) REFERENCES vagas (id)
        )
    ''')

    conn.commit()
    conn.close()

In [ ]:
#2. Integrando ao seu Fluxo de IA
#Agora, em vez de apenas imprimir o resultado, o seu código vai "lembrar" de cada candidato. Isso é muito útil para consultas futuras:

def salvar_candidato_no_banco(vaga_id, dados_ia):
    conn = sqlite3.connect('rh_inteligente.db')
    cursor = conn.cursor()

    cursor.execute('''
        INSERT INTO candidatos (vaga_id, nome, contato, escolaridade, score, resumo_ia, status)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (
        vaga_id,
        dados_ia['nome_candidato'],
        dados_ia['contato'],
        dados_ia['escolaridade_compativel'],
        dados_ia['score_compatibilidade'],
        dados_ia['experiencia_resumo'],
        "Novo"
    ))

    conn.commit()
    conn.close()

In [ ]:
#Arquivo: README.md
# 🤖 NormaIA: Triagem Inteligente de Currículos com IA & Compliance

![Python](https://img.shields.io/badge/python-3.10+-blue.svg)
![Gemini](https://img.shields.io/badge/IA-Google%20Gemini-orange.svg)
![License](https://img.shields.io/badge/License-MIT-green.svg)

O **NormaIA** é um ecossistema de automação para RH que utiliza Inteligência Artificial Generativa para gerir processos seletivos de ponta a ponta. O projeto foi desenhado para unir a eficiência da tecnologia com o rigor técnico da **Segurança do Trabalho (SST)** e a conformidade jurídica do **Direito (LGPD)**.

## 🚀 Funcionalidades Principais

- **📥 Automação de E-mail:** Monitoramento automático de caixas de entrada para captura de currículos em anexo (PDF).
- **🧠 Análise Semântica (Gemini API):** Diferente de filtros comuns, a IA entende o contexto (ex: identifica que um certificado de NR-35 habilita o candidato para "Trabalho em Altura").
- **🛡️ Privacy by Design:** Camada de anonimização via Regex que oculta dados sensíveis (CPF, documentos) antes do processamento pela IA.
- **📊 Dashboard Interativo:** Interface intuitiva em **Streamlit** para o recrutador configurar vagas e visualizar o ranking de candidatos.
- **🗄️ Persistência de Dados:** Armazenamento estruturado em **SQLite** para auditoria e histórico de seleções.

## 🛠️ Tecnologias Utilizadas

- **Linguagem:** Python 3.x
- **IA:** Google Generative AI (Gemini 1.5 Flash)
- **Interface:** Streamlit
- **Banco de Dados:** SQLite
- **Extração de Dados:** Pdfplumber & Re (Regex)
- **Integração:** Imaplib & Email

## 📋 Pré-requisitos

Antes de começar, você precisará das seguintes bibliotecas:

```bash
pip install streamlit google-generativeai pdfplumber pandas gspread oauth2client